
# SQL Analyser un marketplace e-commerce
---

**Dataset** : Olist Brazilian E-Commerce, 100 000 commandes, 9 tables  
**Outil** : DuckDB + jupysql dans Google Colab  

---

### Scénario

Data analyst dans une entreprise d'e-commerce.  
Avant de la réunion mensuelle, le directeur commercial t'envoie un message :

> *"J'ai besoin d'un bilan vendeurs pour ce soir. Chiffre d'affaires, volume de commandes,
> note clients. Le tout en une vue. Et si tu peux me dire lesquels sous-performent, c'est mieux."*



# Téléchargement du dataset dans DuckDB

Lien du dataset: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce/data

Exécutez toutes les cellules dans cette partie pour un setup complet.



In [5]:
%pip install duckdb-engine --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.3 MB/s eta 0:00:00


In [6]:
import sys
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
%matplotlib inline

if 'google.colab' in sys.modules:
    !sudo apt-get update -qq > /dev/null 2>&1
    !pip install -q duckdb > /dev/null 2>&1
    !pip install -q kagglehub > /dev/null 2>&1
    !pip install -q jupysql > /dev/null 2>&1

# Load SQL extension
%load_ext sql

# Config
%config SqlMagic.autopandas = True
pd.options.display.float_format = '{:.2f}'.format

In [7]:
import duckdb
import kagglehub
from pathlib import Path
import time

# Télécharger depuis Kaggle
print("📥 Téléchargement du dataset Kaggle...")
start_time = time.time()
dataset_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

download_time = time.time() - start_time
print(f"✓ Dataset téléchargé en {download_time:.2f}s\n")

# Créer une connexion DuckDB persistante
print("🔌 Création de la base DuckDB...")
con = duckdb.connect('accelerateur-sql.duckdb')
print("✓ Connecté!\n")

# Charger les fichiers CSV
print("📂 Chargement des données...")
csv_files = list(Path(dataset_path).glob('*.csv'))
print(f"📁 {len(csv_files)} fichiers CSV trouvés\n")

total_rows = 0
total_start = time.time()

# Renommer les tables
TABLE_NAMES = {
    'olist_customers_dataset':              'customers',
    'olist_sellers_dataset':                'sellers',
    'olist_orders_dataset':                 'orders',
    'olist_order_items_dataset':            'order_items',
    'olist_order_reviews_dataset':          'order_reviews',
    'olist_order_payments_dataset':         'order_payments',
    'olist_products_dataset':               'products',
    'olist_geolocation_dataset':            'geolocation',
    'product_category_name_translation':    'translation',
}

for idx, csv_file in enumerate(csv_files, 1):
    #table_name = csv_file.stem
    table_name = TABLE_NAMES.get(csv_file.stem, csv_file.stem)

    print(f"[{idx}/{len(csv_files)}] '{table_name}'...", end=" ", flush=True)

    # Charger directement avec DuckDB
    con.execute(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv_auto('{csv_file}')
    """)

    # Compter les lignes
    row_count = con.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    col_count = len(con.execute(f"SELECT * FROM {table_name} LIMIT 1").description)

    print(f"✓ ({row_count:,} lignes, {col_count} colonnes)")
    total_rows += row_count

total_time = time.time() - total_start

print("\n" + "=" * 60)
print(f"✓ COMPLET!")
print(f"  • Tables créées: {len(csv_files)}")
print(f"  • Lignes totales: {total_rows:,}")
print(f"  • Temps total: {total_time:.2f}s")
print("=" * 60)

📥 Téléchargement du dataset Kaggle...
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
✓ Dataset téléchargé en 3.61s

🔌 Création de la base DuckDB...
✓ Connecté!

📂 Chargement des données...
📁 9 fichiers CSV trouvés

[1/9] 'customers'... ✓ (99,441 lignes, 5 colonnes)
[2/9] 'sellers'... ✓ (3,095 lignes, 4 colonnes)
[3/9] 'order_reviews'... ✓ (99,224 lignes, 7 colonnes)
[4/9] 'order_items'... ✓ (112,650 lignes, 7 colonnes)
[5/9] 'products'... ✓ (32,951 lignes, 9 colonnes)
[6/9] 'geolocation'... ✓ (1,000,163 lignes, 5 colonnes)
[7/9] 'translation'... ✓ (71 lignes, 2 colonnes)
[8/9] 'orders'... ✓ (99,441 lignes, 8 colonnes)
[9/9] 'order_payments'... ✓ (103,886 lignes, 5 colonnes)

✓ COMPLET!
  • Tables créées: 9
  • Lignes totales: 1,550,922
  • Temps total: 6.65s


In [8]:
%%sql con

In [9]:
%%sql
-- tester la ligne magique
SELECT * FROM sellers
LIMIT 5

Running query in 'DuckDBPyConnection'

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [10]:
%%sql
SELECT * FROM orders
LIMIT 5

Running query in 'DuckDBPyConnection'

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [11]:
%%sql

SELECT
  *
FROM order_items
LIMIT 5

Running query in 'DuckDBPyConnection'

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [12]:
%%sql

SELECT * FROM order_reviews
LIMIT 5

Running query in 'DuckDBPyConnection'

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,None,None,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,None,None,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,None,None,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,None,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,None,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


## Exercice 1 : Réchauffement

**Contrôle des valeurs manquantes (Data Quality)**

In [32]:
%%sql

SELECT
    COUNT(*) AS total_lignes,
    COUNT(product_category_name) AS categories_renseignees,
    COUNT(*) - COUNT(product_category_name) AS categories_manquantes
FROM products;

Running query in 'DuckDBPyConnection'

,total_lignes,categories_renseignees,categories_manquantes
0,32951,32341,610


**Recherche des doublons**

In [31]:
%%sql

SELECT
    customer_id,
    COUNT(*) AS nb_occurrences
FROM customers
GROUP BY customer_id
HAVING COUNT(*) > 1;

Running query in 'DuckDBPyConnection'

,customer_id,nb_occurrences


**Contrôle de cohérence métier** : Une commande livrée doit avoir une date de livraison.

In [33]:
%%sql

SELECT *
FROM orders
WHERE order_status = 'delivered'
AND order_delivered_customer_date IS NULL;

Running query in 'DuckDBPyConnection'

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
1,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
2,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
3,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
4,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
5,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
6,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
7,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


**Cartographie des données**: Lister les tables reliées

In [34]:
%%sql

SELECT
    o.order_id,
    c.customer_city,
    oi.product_id,
    p.product_category_name
FROM orders o
JOIN customers c
ON o.customer_id = c.customer_id
JOIN order_items oi
ON o.order_id = oi.order_id
JOIN products p
ON oi.product_id = p.product_id
LIMIT 20;

Running query in 'DuckDBPyConnection'

,order_id,customer_city,product_id,product_category_name
0,00010242fe8c5a6d1ba2dd792cb16214,campos dos goytacazes,4244733e06e7ecb4970a6e2683c13e61,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,santa fe do sul,e5f2d52b802189ee658865ca93d83a8f,pet_shop
2,000229ec398224ef6ca0657da4fc703e,para de minas,c777355d18b72b67abbeef9df44fd0fd,moveis_decoracao
3,00024acbcdf0a6daa1e931b038114c75,atibaia,7634da152a4610f1595efa32f14722fc,perfumaria
4,00042b26cf59d7ce69dfabb4e55b4fd9,varzea paulista,ac6c3623068f30de03045865e4e10089,ferramentas_jardim
5,00048cc3ae777c65dbb7d2a0634bc1ea,uberaba,ef92defde845ab8450f9d70c526ef70f,utilidades_domesticas
6,00054e8431b9d7675808bcb819fb4a32,guararapes,8d4f2bb7e93e6710a28f34fa83ee7d28,telefonia
7,000576fe39319847cbb9d288c5617fa6,praia grande,557d850972a7d6f792fd18ae1400d9b6,ferramentas_jardim
8,0005a1a1728c9d785b8e2b08b904576c,santos,310ae3c140ff94b03219ad0adc3c778f,beleza_saude
9,0005f50442cb953dcd1d21e1fb923495,jandira,4535b0e1091c278dfd193e5a1d63b39f,livros_tecnicos


### Quel est le chiffre d'affaires total et le nombre de commandes par vendeur ?

In [13]:
%%sql

SELECT
  seller_id,
  SUM(price) AS total_revenue,
  COUNT(DISTINCT order_id) AS total_orders
FROM order_items
GROUP BY seller_id
ORDER BY total_revenue DESC
LIMIT 20


Running query in 'DuckDBPyConnection'

,seller_id,total_revenue,total_orders
0,4869f7a5dfa277a7dca6462dcf3b52b2,229472.63,1132
1,53243585a1d6dc2643021fd1853d8905,222776.05,358
2,4a3ca9315b744ce9f8e9374361493884,200472.92,1806
3,fa1c13f2614d7b5c4749cbc52fecda94,194042.03,585
4,7c67e1448b00f6e969d365cea6b010ab,187923.89,982
5,7e93a43ef30c4f03f38b393420bc753a,176431.87,336
6,da8622b14eb17ae2831f4ac5b9dab84a,160236.57,1314
7,7a67c85e85bb2ce8582c35f2203ad736,141745.53,1160
8,1025f0e2d44d7041d6cf58b6550e0bfa,138968.55,915
9,955fee9216a65b617aa5c0531780ce60,135171.70,1287


**Catégoriser les données afin de les exploiter au mieux**

In [36]:
%%sql

SELECT
    product_category_name,

    CASE
        WHEN COUNT(*) < 100 THEN 'Faible'
        WHEN COUNT(*) < 1000 THEN 'Moyen'
        ELSE 'Élevé'
    END AS niveau_vente

FROM products p
JOIN order_items oi
ON p.product_id = oi.product_id

GROUP BY product_category_name;

Running query in 'DuckDBPyConnection'

,product_category_name,niveau_vente
0,moveis_decoracao,Élevé
1,perfumaria,Élevé
2,utilidades_domesticas,Élevé
3,beleza_saude,Élevé
4,cama_mesa_banho,Élevé
...,...,...
69,cds_dvds_musicais,Faible
70,moveis_colchao_e_estofado,Faible
71,construcao_ferramentas_ferramentas,Moyen
72,seguros_e_servicos,Faible


## Exercice 2 : Intermédiaire 1
### Vue globale vendeurs : CA, nombre de commandes, note moyenne, avec nom de la ville et de l'Etat

In [14]:
%%sql

--order_reviews pour note moyenne
-- nom de la ville et de l'état: sellers
-- order items pour volume de cde et CA, ont seller id en commun avec order items
-- order reviews et order items ont order id en commun
-- order status dans orders: uniquement des commandes delivered, order id en commun avec order items

SELECT
  sellers.seller_id,
  sellers.seller_city,
  sellers.seller_state,
  SUM(price) AS total_revenue,
  COUNT(DISTINCT order_items.order_id) AS total_orders,
  AVG(order_reviews.review_score) AS avg_review_score
FROM sellers
INNER JOIN order_items
  ON sellers.seller_id = order_items.seller_id
INNER JOIN orders
  ON order_items.order_id = orders.order_id
INNER JOIN order_reviews
  ON orders.order_id = order_reviews.order_id
WHERE orders.order_status = 'delivered'
GROUP BY ALL
ORDER BY total_revenue DESC
LIMIT 20


Running query in 'DuckDBPyConnection'

,seller_id,seller_city,seller_state,total_revenue,total_orders,avg_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,225586.34,1116,4.14
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,215904.44,346,4.13
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,197225.32,1753,3.83
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,189649.54,574,4.37
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,186664.01,967,3.35
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,165751.50,318,4.36
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,161574.27,1305,4.08
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,139188.73,1137,4.27
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,138691.40,902,3.87
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,130823.82,1253,4.09


In [15]:
%%sql
-- sanity check
-- combien de lignes par order id dans order items
SELECT
  COUNT(DISTINCT order_id) AS distinct_orders,
  COUNT(*) AS total_rows
FROM order_items


Running query in 'DuckDBPyConnection'

,distinct_orders,total_rows
0,98666,112650


In [16]:
%%sql
-- nb de lignes par order id dans order_reviews

SELECT
  COUNT(DISTINCT order_id) AS distinct_orders,
  COUNT(*) AS total_rows
FROM order_reviews

Running query in 'DuckDBPyConnection'

,distinct_orders,total_rows
0,98673,99224


In [17]:
%%sql
--combien de reviews par commande

SELECT
  order_id,
  COUNT(*)  AS review_count
FROM order_reviews
GROUP BY order_id
ORDER BY review_count DESC
LIMIT 10


Running query in 'DuckDBPyConnection'

,order_id,review_count
0,c88b1d1b157a9999ce368f218a407141,3
1,8e17072ec97ce29f0e1f111e598b0c85,3
2,df56136b8031ecd28e200bb18e6ddb2e,3
3,03c939fd7fd3b38f8485a0f95798f1f6,3
4,169d7e0fd71d624d306f132acd791cbe,2
5,0e457aee274ec7e2e18d25ab6d212921,2
6,5040757d4e06a4be96d3827b860b4e7c,2
7,f0fbc60d51bb40c156688d9ce008237f,2
8,3df55fc07ff463109ce0422439693aee,2
9,e832fe7f7808b5f981ee02746681d40e,2


In [18]:
%%sql

SELECT
  sellers.seller_id,
  sellers.seller_city,
  sellers.seller_state,
  price,
  order_items.order_id,
  order_reviews.review_score
FROM sellers
INNER JOIN order_items
  ON sellers.seller_id = order_items.seller_id
INNER JOIN orders
  ON order_items.order_id = orders.order_id
INNER JOIN order_reviews
  ON orders.order_id = order_reviews.order_id
WHERE orders.order_status = 'delivered'
AND order_items.order_id = 'c88b1d1b157a9999ce368f218a407141'

Running query in 'DuckDBPyConnection'

,seller_id,seller_city,seller_state,price,order_id,review_score
0,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,34.99,c88b1d1b157a9999ce368f218a407141,3
1,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,34.99,c88b1d1b157a9999ce368f218a407141,5
2,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,34.99,c88b1d1b157a9999ce368f218a407141,5


In [19]:
%%sql
SELECT * FROM order_items
WHERE order_id = 'c88b1d1b157a9999ce368f218a407141'


Running query in 'DuckDBPyConnection'

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,c88b1d1b157a9999ce368f218a407141,1,b1acb7e8152c90c9619897753a75c973,cc419e0650a3c5ba77189a1882b7556a,2017-07-26 22:50:12,34.99,7.78


In [20]:
%%sql
SELECT * FROM order_reviews
WHERE order_id = 'c88b1d1b157a9999ce368f218a407141'


Running query in 'DuckDBPyConnection'

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,None,None,2017-07-22,2017-07-26 13:41:07
1,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,None,None,2017-07-22,2017-07-26 13:40:22
2,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,None,None,2017-07-21,2017-07-26 13:45:15


In [21]:
%%sql
-- pré agréger avant de joindre

WITH order_value AS (
  SELECT
    order_id,
    seller_id,
    SUM(price) AS order_revenue
  FROM order_items
  GROUP BY ALL
)

, order_avg_reviews AS (
  SELECT
    order_id,
    AVG(review_score) AS avg_review_score
  FROM order_reviews
  GROUP BY ALL
)

SELECT
  sellers.seller_id,
  sellers.seller_city,
  sellers.seller_state,
  SUM(order_value.order_revenue) AS total_revenue,
  COUNT(DISTINCT order_value.order_id) AS total_orders,
  AVG(order_avg_reviews.avg_review_score) AS avg_review_score
FROM sellers
INNER JOIN order_value
  ON sellers.seller_id = order_value.seller_id
INNER JOIN orders
  ON order_value.order_id = orders.order_id
INNER JOIN order_avg_reviews
  ON orders.order_id = order_avg_reviews.order_id
WHERE orders.order_status = 'delivered'
GROUP BY ALL
ORDER BY total_revenue DESC
LIMIT 20


Running query in 'DuckDBPyConnection'

,seller_id,seller_city,seller_state,total_revenue,total_orders,avg_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,225586.34,1116,4.15
1,53243585a1d6dc2643021fd1853d8905,lauro de freitas,BA,215904.44,346,4.19
2,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,194699.12,1753,3.85
3,fa1c13f2614d7b5c4749cbc52fecda94,sumare,SP,189649.54,574,4.37
4,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,185170.23,967,3.50
5,7e93a43ef30c4f03f38b393420bc753a,barueri,SP,165751.50,318,4.36
6,da8622b14eb17ae2831f4ac5b9dab84a,piracicaba,SP,159087.47,1305,4.18
7,7a67c85e85bb2ce8582c35f2203ad736,sao paulo,SP,138608.77,1137,4.27
8,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,137179.80,902,4.01
9,955fee9216a65b617aa5c0531780ce60,sao paulo,SP,130753.82,1253,4.20


## Exercice 3 : Intermédiaire 2
### Vendeurs à risque : note sous la moyenne, volume au-dessus de la moyenne

In [22]:
%%sql

-- correction
/*
Un vendeur avec une mauvaise note et 3 commandes, ce n'est pas urgent.
Un vendeur avec une mauvaise note et 200 commandes, c'est un vrai problème.

On ajoute une deuxième CTE `global_benchmark` pour calculer les moyennes globales,
et on compare chaque vendeur à ces seuils.
*/



WITH order_value AS (
    -- pré agréger order_items à la maille order_id et seller_id avant de joindre
    SELECT
        order_id,
        seller_id,
        SUM(price) AS order_revenue
    FROM order_items
    GROUP BY order_id, seller_id
),
seller_orders AS (
    SELECT
        sellers.seller_id,
        sellers.seller_city,
        sellers.seller_state,
        order_value.order_id,
        order_value.order_revenue
    FROM sellers
    INNER JOIN order_value
      ON sellers.seller_id = order_value.seller_id
    INNER JOIN orders
      ON order_value.order_id = orders.order_id
    WHERE orders.order_status = 'delivered'
),
order_avg_review AS (
    SELECT
        order_id,
        AVG(review_score) AS avg_review_score
    FROM order_reviews
    GROUP BY order_id
),
seller_performance AS (
SELECT
    seller_orders.seller_id,
    seller_orders.seller_city,
    seller_orders.seller_state,
    COUNT(seller_orders.order_id) AS total_orders,
    ROUND(SUM(seller_orders.order_revenue), 2) AS total_revenue,
    ROUND(AVG(order_avg_review.avg_review_score), 2) AS avg_review_score
FROM seller_orders
LEFT JOIN order_avg_review
  ON seller_orders.order_id = order_avg_review.order_id
GROUP BY ALL
-- ORDER BY total_revenue DESC
),
global_avg AS (
  SELECT
      ROUND(AVG(avg_review_score), 2) AS global_avg_score,
      ROUND(AVG(total_orders), 0) AS global_avg_orders
  FROM seller_performance
)

-- SELECT * FROM global_avg

-- Le CROSS JOIN
SELECT
  seller_id,
  seller_city,
  seller_state,
  total_orders,
  total_revenue,
  avg_review_score,
  global_avg.global_avg_score,
  global_avg.global_avg_orders
FROM seller_performance
CROSS JOIN global_avg
WHERE avg_review_score < global_avg.global_avg_score
  AND total_orders > global_avg.global_avg_orders
ORDER BY total_revenue DESC
LIMIT 20


Running query in 'DuckDBPyConnection'

,seller_id,seller_city,seller_state,total_orders,total_revenue,avg_review_score,global_avg_score,global_avg_orders
0,4869f7a5dfa277a7dca6462dcf3b52b2,guariba,SP,1124,226987.93,4.15,4.18,33.00
1,4a3ca9315b744ce9f8e9374361493884,ibitinga,SP,1772,196882.12,3.85,4.18,33.00
2,7c67e1448b00f6e969d365cea6b010ab,itaquaquecetuba,SP,973,186570.05,3.50,4.18,33.00
3,1025f0e2d44d7041d6cf58b6550e0bfa,sao paulo,SP,910,138208.56,4.01,4.18,33.00
4,6560211a19b47992c3666cc44a7e94c0,sao paulo,SP,1819,120702.83,3.98,4.18,33.00
5,7d13fca15225358621be4086e1eb0964,ribeirao preto,SP,558,112436.18,4.04,4.18,33.00
6,5dceca129747e92ff8ef7a997dc4f8ca,santa barbara d´oeste,SP,322,111126.73,4.01,4.18,33.00
7,1f50f920176fa81dab994f9023523100,sao jose do rio preto,SP,1399,106655.71,4.14,4.18,33.00
8,cc419e0650a3c5ba77189a1882b7556a,santo andre,SP,1651,101090.96,4.15,4.18,33.00
9,cca3071e3e9bb7d12640c9fbe2301306,ibitinga,SP,699,63035.16,3.88,4.18,33.00


## Exercice 4 : Avancé
### Top 3 vendeurs par catégorie de produits


In [23]:
%%sql

SELECT * FROM products
LIMIT 5

Running query in 'DuckDBPyConnection'

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
3,cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


In [24]:
%%sql
SELECT * FROM translation
LIMIT 5

Running query in 'DuckDBPyConnection'

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [25]:
%%sql


WITH order_value AS (
  SELECT
    order_id,
    seller_id,
    product_id,
    SUM(price) AS order_revenue
  FROM order_items
  GROUP BY ALL
)

, seller_performance AS (
SELECT
  sellers.seller_id,
  sellers.seller_city,
  sellers.seller_state,
  products.product_category_name,
  SUM(order_value.order_revenue) AS total_revenue
FROM order_value
INNER JOIN sellers
  ON sellers.seller_id = order_value.seller_id
INNER JOIN orders
  ON order_value.order_id = orders.order_id
INNER JOIN products
  ON order_value.product_id = products.product_id
WHERE orders.order_status = 'delivered'
GROUP BY ALL
ORDER BY sellers.seller_id,
  sellers.seller_city,
  sellers.seller_state

)

,  rank_seller AS (
SELECT
  seller_id,
  seller_city,
  seller_state,
  product_category_name,
  total_revenue,
  ROW_NUMBER() OVER (
    PARTITION BY product_category_name
    ORDER BY total_revenue DESC) AS revenue_rank
FROM seller_performance
ORDER BY product_category_name,
  total_revenue DESC
)

SELECT *
FROM rank_seller
WHERE revenue_rank <= 3
ORDER BY product_category_name,
  total_revenue DESC

Running query in 'DuckDBPyConnection'

,seller_id,seller_city,seller_state,product_category_name,total_revenue,revenue_rank
0,e59aa562b9f8076dd550fcddf0e73491,curitiba,PR,agro_industria_e_comercio,30716.30,1
1,6bd69102ab48df500790a8cecfc285c2,sao paulo,SP,agro_industria_e_comercio,8070.00,2
2,f08a5b9dd6767129688d001acafc21e5,porto alegre,RS,agro_industria_e_comercio,7535.54,3
3,cbd996ad3c1b7dc71fd0e5f5df9087e2,sao jose do rio preto,SP,alimentos,4915.71,1
4,d13e50eaa47b4cbe9eb81465865d8cfc,santo andre,SP,alimentos,4822.62,2
...,...,...,...,...,...,...
212,53e4c6e0f4312d4d2107a8c9cddf45cd,pedreira,SP,utilidades_domesticas,26790.34,2
213,9de4643a8dbde634fe55621059d92273,joinville,SC,utilidades_domesticas,17294.41,3
214,c826c40d7b19f62a09e2d7c5e7295ee2,guarulhos,SP,None,44542.27,1
215,5d378b73ab7dd6f0418d743e5dcb0bd1,sao paulo,SP,None,11254.00,2


In [26]:
%%sql

-- QUALIFY pour un code plus élégant
WITH order_value AS (
  SELECT
    order_id,
    seller_id,
    product_id,
    SUM(price) AS order_revenue
  FROM order_items
  GROUP BY ALL
)

, seller_performance AS (
SELECT
  sellers.seller_id,
  sellers.seller_city,
  sellers.seller_state,
  products.product_category_name,
  SUM(order_value.order_revenue) AS total_revenue
FROM order_value
INNER JOIN sellers
  ON sellers.seller_id = order_value.seller_id
INNER JOIN orders
  ON order_value.order_id = orders.order_id
INNER JOIN products
  ON order_value.product_id = products.product_id
WHERE orders.order_status = 'delivered'
GROUP BY ALL
ORDER BY sellers.seller_id,
  sellers.seller_city,
  sellers.seller_state

)


SELECT
  seller_id,
  seller_city,
  seller_state,
  product_category_name,
  total_revenue
FROM seller_performance
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_category_name
    ORDER BY total_revenue DESC) <= 3
ORDER BY product_category_name,
  total_revenue DESC


Running query in 'DuckDBPyConnection'

,seller_id,seller_city,seller_state,product_category_name,total_revenue
0,e59aa562b9f8076dd550fcddf0e73491,curitiba,PR,agro_industria_e_comercio,30716.30
1,6bd69102ab48df500790a8cecfc285c2,sao paulo,SP,agro_industria_e_comercio,8070.00
2,f08a5b9dd6767129688d001acafc21e5,porto alegre,RS,agro_industria_e_comercio,7535.54
3,cbd996ad3c1b7dc71fd0e5f5df9087e2,sao jose do rio preto,SP,alimentos,4915.71
4,d13e50eaa47b4cbe9eb81465865d8cfc,santo andre,SP,alimentos,4822.62
...,...,...,...,...,...
212,53e4c6e0f4312d4d2107a8c9cddf45cd,pedreira,SP,utilidades_domesticas,26790.34
213,9de4643a8dbde634fe55621059d92273,joinville,SC,utilidades_domesticas,17294.41
214,c826c40d7b19f62a09e2d7c5e7295ee2,guarulhos,SP,None,44542.27
215,5d378b73ab7dd6f0418d743e5dcb0bd1,sao paulo,SP,None,11254.00


**Exercise 5 Analyse des écarts**

In [27]:
%%sql

SELECT
    order_status,
    COUNT(*) AS total_commandes
FROM orders
GROUP BY order_status
ORDER BY total_commandes DESC;

Running query in 'DuckDBPyConnection'

,order_status,total_commandes
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


**La cartographie des données dans Draw.io :**


In [28]:
CUSTOMERS
      │
      ▼
ORDERS
      │
      ▼
ORDER_ITEMS
 ┌────┴─────┐
 ▼          ▼
PRODUCTS   SELLERS
      │
      ▼
ORDER_PAYMENTS

Ah d'accord 😊. Tu veux un **exercice Visio / Draw.io basé sur ton projet e-commerce (Olist)** et non sur DGFiP ou POK.

Pour ton notebook SQL e-commerce, voici la cartographie des données que je mettrais dans Draw.io :

```text
CUSTOMERS
(customer_id)
      │
      │
      ▼
ORDERS
(order_id)
(customer_id)
(order_status)
      │
      │
 ┌────┴────┐
 │         │
 ▼         ▼

ORDER_ITEMS      ORDER_PAYMENTS
(order_id)       (order_id)
(product_id)     (payment_type)
(seller_id)      (payment_value)
(price)

 │
 │
 ├──────────► PRODUCTS
 │             (product_id)
 │             (category)
 │
 ▼

SELLERS
(seller_id)
(seller_state)
```

---

### Exercice SQL 1 : Cartographie ERP

Lister toutes les informations d'une commande :

```sql
SELECT
    o.order_id,
    c.customer_city,
    o.order_status,
    p.product_category_name,
    s.seller_state,
    op.payment_type,
    op.payment_value
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
JOIN products p
    ON oi.product_id = p.product_id
JOIN sellers s
    ON oi.seller_id = s.seller_id
JOIN order_payments op
    ON o.order_id = op.order_id
LIMIT 20;
```

👉 Cette requête montre la circulation de la donnée dans le système.

---

### Exercice SQL 2 : Contrôle qualité

Produits sans catégorie :

```sql
SELECT *
FROM products
WHERE product_category_name IS NULL;
```

---

### Exercice SQL 3 : Analyse des écarts

Statuts des commandes :

```sql
SELECT
    order_status,
    COUNT(*) AS total_commandes
FROM orders
GROUP BY order_status
ORDER BY total_commandes DESC;
```

Puis poser la question :

> Pourquoi certaines commandes sont-elles annulées alors que d'autres sont livrées ?

---

### Exercice SQL 4 : Supply Chain

Temps moyen de livraison par État :

```sql
SELECT
    c.customer_state,
    ROUND(
        AVG(
            JULIANDAY(o.order_delivered_customer_date)
            - JULIANDAY(o.order_purchase_timestamp)
        ),
        2
    ) AS delai_moyen
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY delai_moyen DESC;
```

---

### Ce que tu peux dessiner dans Draw.io

```text
Client
   │
   ▼
Commande
   │
   ▼
Paiement
   │
   ▼
Produit
   │
   ▼
Vendeur
   │
   ▼
Livraison
```

ou plus détaillé :

```text
CUSTOMERS
      │
      ▼
ORDERS
      │
      ▼
ORDER_ITEMS
 ┌────┴─────┐
 ▼          ▼
PRODUCTS   SELLERS
      │
      ▼
ORDER_PAYMENTS
```


"La donnée client alimente la commande, qui génère un paiement et des lignes de commande. Ces lignes sont reliées aux produits et aux vendeurs. Cette cartographie permet de comprendre l'origine des données, les relations entre les tables et les impacts d'une anomalie sur le système."
